# Mercury - Pro Tier Fine-Tuning (Apollo 1.2)

Covers Apollo's two engines: Whisper-Large-v3-Turbo (ASR) and Bark (TTS).
Full fine-tune and LoRA/PEFT paths for each - see MODELVERSION.md for what a fine-tune does to the version number (1.2 -> 2.0 on first fine-tune, then 2.0 -> 2.1 -> 2.2 for each one after).

## 0. Setup - install dependencies

In [ ]:
!pip install -q transformers==4.47.1 torch==2.5.1 peft accelerate soundfile pydub jiwer huggingface_hub datasets

## 1. Convert any WAV source audio to MP3

Run once against your raw dataset directory before building manifests - keeps disk
quota and any exported archives small. Skip if your data is already MP3.

In [ ]:
import os
from pydub import AudioSegment

DATASET_DIR = "/kaggle/input/your-dataset"  # <-- point at your real dataset
OUT_DIR = "/kaggle/working/audio_mp3"
os.makedirs(OUT_DIR, exist_ok=True)

converted = 0
for root, _, files in os.walk(DATASET_DIR):
    for f in files:
        if f.lower().endswith(".wav"):
            src = os.path.join(root, f)
            dst = os.path.join(OUT_DIR, os.path.splitext(f)[0] + ".mp3")
            AudioSegment.from_wav(src).export(dst, format="mp3", bitrate="128k")
            converted += 1
print(f"Converted {converted} wav files to mp3 in {OUT_DIR}")

## 2. Build train/val manifests

NeMo-format manifests (audio_filepath/text/duration per line) - built once, reused by
every model below.

In [ ]:
import json
import soundfile as sf

MANIFEST_TRAIN = "/kaggle/working/train_manifest.json"
MANIFEST_VAL = "/kaggle/working/val_manifest.json"

def build_manifest(audio_text_pairs, out_path):
    with open(out_path, "w") as f:
        for audio_path, text in audio_text_pairs:
            info = sf.info(audio_path)
            f.write(json.dumps({
                "audio_filepath": audio_path,
                "text": text,
                "duration": info.duration,
            }) + "\n")

# TODO: populate from your real (audio_path, transcript) pairs once data is ready.
train_pairs = []
val_pairs = []
build_manifest(train_pairs, MANIFEST_TRAIN)
build_manifest(val_pairs, MANIFEST_VAL)
print(f"train={len(train_pairs)} val={len(val_pairs)} examples (0 until real data is wired in)")

---
## 3. ASR - Whisper-Large-v3-Turbo (LoRA/PEFT path)

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model

MODEL_ID = "openai/whisper-large-v3-turbo"
processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### 3b. ASR - full fine-tune path (no LoRA)

Needs meaningfully more VRAM/time than LoRA on this model size - only use if LoRA
underperforms on your dataset.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL_ID = "openai/whisper-large-v3-turbo"
processor_full = WhisperProcessor.from_pretrained(MODEL_ID)
model_full = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model_full.train()
print(f"full fine-tune: {sum(p.numel() for p in model_full.parameters() if p.requires_grad):,} trainable params")

---
## 4. TTS - Bark (LoRA/PEFT path)

Bark's semantic/coarse/fine sub-models each need their own LoRA target modules -
start with the semantic model only, since it's the one that carries linguistic content.

In [ ]:
from transformers import AutoProcessor, BarkModel
from peft import LoraConfig, get_peft_model

MODEL_ID = "suno/bark"
bark_processor = AutoProcessor.from_pretrained(MODEL_ID)
bark_model = BarkModel.from_pretrained(MODEL_ID)

lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["c_attn"], lora_dropout=0.05, bias="none")
bark_model.semantic = get_peft_model(bark_model.semantic, lora_config)
bark_model.semantic.print_trainable_parameters()

---
## 5. Evaluation harness (run after any fine-tune, before promoting a model)

In [ ]:
from jiwer import wer, cer

def evaluate(model_transcribe_fn, val_pairs):
    refs, hyps = [], []
    for audio_path, ref_text in val_pairs:
        hyps.append(model_transcribe_fn(audio_path))
        refs.append(ref_text)
    if not refs:
        return {"wer": None, "cer": None, "n": 0}
    return {"wer": wer(refs, hyps), "cer": cer(refs, hyps), "n": len(refs)}

# evaluate(lambda path: ..., val_pairs)  # wire up the real transcribe fn once training runs